In [2]:
import math
import numpy as np
import scipy.stats as st
import scipy.optimize as opt

rng = np.random.default_rng(52)

ALPHA = 0.05
K = 10
N = 100
freq = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])
bins = np.arange(K)

F_emp = np.concatenate([[0], np.cumsum(freq)]) / N


def kolmogorov_pvalue(delta):
    total = 1.0
    for k in range(1, 1001):
        total += 2 * (-1)**k * np.exp(-2 * k**2 * delta**2)
    return 1.0 - total


def norm_cdf(x, mu, sigma):
    return 0.5 * (1 + math.erf((x - mu) / (math.sqrt(2) * sigma)))


def verdict(p):
    return 'нет оснований для отклонения гипотезы' if p > ALPHA else 'отвергается'


expected_unif = np.full(K, N / K)
T_chi_unif = float(np.sum((freq - expected_unif)**2 / expected_unif))
p_chi_unif = float(st.chi2.sf(T_chi_unif, df=K - 1))

print('Равномерная модель R(0,10), критерий хи-квадрат')
print(f'chi^2 = {T_chi_unif:.6f}; p-value = {p_chi_unif:.4f}')
print('H0:', verdict(p_chi_unif))

F_unif = bins / K
D_unif = max(
    max(abs(F_emp[i] - F_unif[i]), abs(F_emp[i + 1] - F_unif[i]))
    for i in range(K)
)
delta_unif = math.sqrt(N) * D_unif
p_ks_unif = kolmogorov_pvalue(delta_unif)

print()
print('Равномерная модель R(0,10), критерий Колмогорова')
print(f'Δ = {delta_unif:.2f}; p-value = {p_ks_unif:.4f}')
print('H0:', verdict(p_ks_unif))


def neg_loglik(params):
    mu, sigma = params
    cdf = st.norm.cdf(bins[1:], loc=mu, scale=sigma)
    p = np.empty(K)
    p[0] = cdf[0]
    p[1:-1] = np.diff(cdf)
    p[-1] = 1.0 - cdf[-1]
    p = np.clip(p, 1e-15, None)
    return -float(np.dot(freq, np.log(p)))


result = opt.differential_evolution(neg_loglik, bounds=[(0, 10), (0.1, 10)], maxiter=10000, seed=0)
mu_mle, sigma_mle = result.x

cdf_mle = st.norm.cdf(bins[1:], loc=mu_mle, scale=sigma_mle)
exp_normal = np.empty(K)
exp_normal[0] = N * cdf_mle[0]
exp_normal[1:-1] = N * np.diff(cdf_mle)
exp_normal[-1] = N * (1.0 - cdf_mle[-1])

T_chi_norm = float(np.sum((freq - exp_normal)**2 / exp_normal))
p_chi_norm = float(st.chi2.sf(T_chi_norm, df=K - 3))

print()
print('Нормальная модель N(theta1, theta2^2), критерий хи-квадрат')
print(f'theta1 = {mu_mle:.6f}; theta2 = {sigma_mle:.6f}')
print(f'chi^2 = {T_chi_norm:.6f}; p-value = {p_chi_norm:.4f}')
print('H0:', verdict(p_chi_norm))

sample = np.repeat(bins, freq).astype(float)
mu_hat = sample.mean()
sigma_hat = sample.std(ddof=1)

D_norm = max(
    math.sqrt(N) * max(
        abs(norm_cdf(bins[i], mu_hat, sigma_hat) - F_emp[i]),
        abs(norm_cdf(bins[i], mu_hat, sigma_hat) - F_emp[i + 1])
    )
    for i in range(K)
)

NBOOT = 50000


def bootstrap_ks(B):
    dist = st.norm(loc=mu_hat, scale=sigma_hat)
    results = []
    for _ in range(B):
        s = np.sort(dist.rvs(size=N))
        m_b = s.mean()
        s_b = s.std(ddof=1)
        F_b = np.arange(N + 1) / N
        stat = max(
            math.sqrt(N) * max(
                abs(norm_cdf(s[j], m_b, s_b) - F_b[j]),
                abs(norm_cdf(s[j], m_b, s_b) - F_b[j + 1])
            )
            for j in range(N)
        )
        results.append(stat)
    return results


boot_vals = bootstrap_ks(NBOOT)
p_ks_norm = float(np.mean(np.array(boot_vals) >= D_norm))

print()
print('Нормальная модель N(theta1, theta2^2), критерий Колмогорова')
print(f'Δ = {D_norm:.6f}')
print(f'p-value: {p_ks_norm:.4f}')
print('H0:', verdict(p_ks_norm))

Равномерная модель R(0,10), критерий хи-квадрат
chi^2 = 16.400000; p-value = 0.0590
H0: нет оснований для отклонения гипотезы

Равномерная модель R(0,10), критерий Колмогорова
Δ = 1.40; p-value = 0.0397
H0: отвергается

Нормальная модель N(theta1, theta2^2), критерий хи-квадрат
theta1 = 5.289677; theta2 = 2.679520
chi^2 = 9.802554; p-value = 0.2000
H0: нет оснований для отклонения гипотезы

Нормальная модель N(theta1, theta2^2), критерий Колмогорова
Δ = 1.002094
p-value: 0.0145
H0: отвергается
